# GPU Final Notebook - Math Carry

Modernized final arithmetic run using behavior-specific math SAEs plus the current contrast-target, all-position swap, layer-scan, and graph-positive sparse intervention methods.

This notebook trains or resumes final addition-specific SAEs in `sae_checkpoints_math_final_train`, then saves distinct `math_final_*` graph/intervention outputs.


## Step 1 - Mount Drive and fetch code


In [ ]:
# Step 1: Mount Google Drive and fetch project code
# GitHub is the primary source for code. Drive project.zip is kept as a backup.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = "https://github.com/evey-dev/test_run.git"
repo_dir = "/content/test_run"
use_github_code = True

def run_cmd(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

github_ok = False
if use_github_code:
    try:
        clone_dir = repo_dir
        if os.path.isdir(os.path.join(clone_dir, ".git")):
            run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
        else:
            if os.path.exists(clone_dir) and os.listdir(clone_dir):
                clone_dir = "/content/test_run_github"
            if os.path.isdir(os.path.join(clone_dir, ".git")):
                run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
            elif os.path.exists(clone_dir) and os.listdir(clone_dir):
                raise RuntimeError(f"{clone_dir} exists but is not a git repo")
            else:
                run_cmd(["git", "clone", "--depth", "1", repo_url, clone_dir])
        os.chdir(clone_dir)
        github_ok = True
        print(f"Using GitHub checkout: {os.getcwd()}")
    except Exception as exc:
        print("GitHub checkout failed; falling back to Drive project.zip.")
        print(repr(exc))

if not github_ok:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Could not find {zip_path}. Fix GitHub access or upload project.zip.")
    print(f"Extracting backup zip {zip_path}...")
    run_cmd(["unzip", "-q", "-o", zip_path, "-d", "/content/"])
    for candidate in ["/content/test_run", "/content/mphil_project/test_run", "/content"]:
        if os.path.isdir(os.path.join(candidate, "src")) and os.path.isdir(os.path.join(candidate, "configs")):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError("Could not find extracted project root containing src/ and configs/.")
    print(f"Using Drive backup project: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
!ls -l


## Step 2 - Install dependencies


In [ ]:
# Step 2: Install dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .


## Step 3 - Generate addition dataset


In [ ]:
# Step 3: Generate addition dataset
!python data/generate_datasets.py
print("Dataset files:")
!ls -lh data/addition_data.csv


## Step 4 - Capture math-only activations


In [ ]:
# Capture addition-only MLP activations for final math SAE training
!python src/capture_activations.py \
  --output-dir mechanistic_data_math_final_train \
  --behaviours addition \
  --layers 4 8 12 16 20 24 28 \
  --seed 787

print("Final math activation bundle:")
!ls -lh mechanistic_data_math_final_train


## Step 5 - Train final math-specific SAEs


In [ ]:
# Train final addition-specific SAEs for all selected layers
!python src/train.py \
  --config configs/sae_math_final_train_config.yaml \
  --drive-dir /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_math_final_train

print("Final math local checkpoints:")
!ls -lh mechanistic_data/sae_checkpoints_math_final_train
print("Final math Drive checkpoints:")
!ls -lh /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_math_final_train


## Step 6 - Sanity check the carry/no-carry minimal pair


In [ ]:
# Confirm token positions for the two prompts before all-position swaps.
# The prompts should differ only at the first operand's ones digit: 54 vs 58.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --print-tokens

!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 54 + 83? Answer: 1" \
  --print-tokens


## Step 7 - Rebuild contrast attribution graphs


In [ ]:
# Carry graph: 58 + 83 = 141, tens digit should be 4 rather than 3.
!python src/attribution_graph.py \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target "4" \
  --contrast-target "3" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/math_final_carry_58_83_4v3_graph.json \
  --output-html outputs/math_final_carry_58_83_4v3_graph.html \
  --output-mermaid outputs/math_final_carry_58_83_4v3_graph.md


In [ ]:
# No-carry graph: 54 + 83 = 137, tens digit should be 3 rather than 4.
!python src/attribution_graph.py \
  --prompt "Question: What is 54 + 83? Answer: 1" \
  --target "3" \
  --contrast-target "4" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/math_final_nocarry_54_83_3v4_graph.json \
  --output-html outputs/math_final_nocarry_54_83_3v4_graph.html \
  --output-mermaid outputs/math_final_nocarry_54_83_3v4_graph.md


In [ ]:
# Persist final math graphs immediately, because graph construction is the expensive part.
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/math_final_*_graph.*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')


## Step 8 - Component-level controls


In [ ]:
# All-position MLP knockout on the carry prompt.
# This is the strongest math diagnostic from the previous notebook.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 2" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --full-knockout \
  --knockout-component mlp \
  --positions all \
  --layer-scan \
  --output outputs/math_final_knockout_mlp_allpos_carry_58_83.json


In [ ]:
# Progressive sparse ablation of graph-positive carry features.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --target-token "4, 3, 2" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --graph-json outputs/math_final_carry_58_83_4v3_graph.json \
  --graph-feature-sign positive \
  --scan \
  --positions all \
  --output outputs/math_final_scan_carry_positive_allpos.json


## Step 9 - Full latent minimal-pair swaps


In [ ]:
# Patch NO-CARRY (54+83 -> tens 3) into CARRY (58+83 -> tens 4).
# Desired causal direction: 4 falls and 3 rises.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 54 + 83? Answer: 1" \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --target-token "4, 3, 2" \
  --positions all \
  --layer-scan \
  --output outputs/math_final_swap_full_nocarry_to_carry.json


In [ ]:
# Reverse direction: patch CARRY (58+83 -> tens 4) into NO-CARRY (54+83 -> tens 3).
# Desired causal direction: 3 falls and 4 rises.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 58 + 83? Answer: 1" \
  --prompt "Question: What is 54 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --target-token "4, 3, 2" \
  --positions all \
  --layer-scan \
  --output outputs/math_final_swap_full_carry_to_nocarry.json


## Step 10 - Sparse graph-feature swaps


In [ ]:
# Sparse version of NO-CARRY -> CARRY, using no-carry graph-positive features.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 54 + 83? Answer: 1" \
  --prompt "Question: What is 58 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --graph-json outputs/math_final_nocarry_54_83_3v4_graph.json \
  --graph-feature-sign positive \
  --target-token "4, 3, 2" \
  --positions all \
  --output outputs/math_final_swap_sparse_nocarry_to_carry.json


In [ ]:
# Sparse reverse: CARRY -> NO-CARRY, using carry graph-positive features.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Question: What is 58 + 83? Answer: 1" \
  --prompt "Question: What is 54 + 83? Answer: 1" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_math_final_train_config.yaml \
  --graph-json outputs/math_final_carry_58_83_4v3_graph.json \
  --graph-feature-sign positive \
  --target-token "4, 3, 2" \
  --positions all \
  --output outputs/math_final_swap_sparse_carry_to_nocarry.json


## Step 11 - Persist final math outputs


In [ ]:
# Copy final math graphs/results to Drive
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/math_final_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')
